# Building a Robust HMM Part-of-Speech Tagger
**Natural Language Processing — Mini Project**

This notebook implements a Hidden Markov Model (HMM) POS tagger **from scratch**
(no NLTK taggers, no `hmmlearn`) and trains/evaluates it on the **Brown Corpus**
with the **universal tagset** (12 tags).

It solves the two real-world engineering hurdles called out in the brief:
1. **Out-of-Vocabulary (OOV) words** — handled with **Laplace (add-α) smoothing** and a dedicated `<OOV>` probability mass.
2. **Computational underflow** — handled by rewriting Viterbi decoding in **log-space**.

`numpy` is used only for fast array arithmetic in the Viterbi DP table; all HMM
logic (counts, smoothing, recurrence, backtracking) is hand-written.

In [1]:
import random
from collections import defaultdict, Counter
import numpy as np
import nltk
from nltk.corpus import brown

nltk.download('brown', quiet=True)
nltk.download('universal_tagset', quiet=True)

# log(0) is undefined. Adding this tiny constant to every probability before
# taking the log turns -inf into a large finite negative number (soft -inf).
EPS = 1e-12

## Step 1 — Data Acquisition & Splitting
We split at the **sentence level** (not the word level) so each sentence's
Markov chain stays intact. A fixed seed makes the 80/20 split reproducible.

In [2]:
def load_data(seed=42, train_frac=0.80):
    sents = list(brown.tagged_sents(tagset='universal'))
    rng = random.Random(seed)
    rng.shuffle(sents)
    cut = int(len(sents) * train_frac)
    return sents[:cut], sents[cut:]

train_sents, test_sents = load_data()
print(f"train sentences: {len(train_sents):,}")
print(f"test  sentences: {len(test_sents):,}")
print("example:", train_sents[0][:6])

train sentences: 45,872
test  sentences: 11,468
example: [('He', 'PRON'), ('let', 'VERB'), ('her', 'PRON'), ('tell', 'VERB'), ('him', 'PRON'), ('all', 'PRT')]


## Step 2 — Parameter Estimation (MLE) + Laplace Smoothing

We estimate the three HMM parameter sets by **Maximum Likelihood Estimation**
(relative frequency counts):

- **Initial** $\pi(s) = \dfrac{\#\{\text{sentences starting with }s\}}{\#\text{sentences}}$
- **Transition** $A_{ij}=P(s_j\mid s_i)=\dfrac{\text{Count}(s_i\rightarrow s_j)}{\text{Count}(s_i)}$
- **Emission** $B$, smoothed with **Laplace add-α**:

$$P_{\text{smoothed}}(o_t\mid s_i)=\frac{\text{Count}(o_t\text{ tagged }s_i)+\alpha}{\text{Count}(s_i)+\alpha|V|}$$

A dedicated `"<OOV>"` key stores the leftover mass $\frac{\alpha}{\text{Count}(s_i)+\alpha|V|}$
for words never seen in training. We store **log** parameters directly.

In [3]:
def train_hmm(train_sents, alpha=1.0):
    init_counts  = Counter()             # tag at sentence position 0
    trans_counts = defaultdict(Counter)  # tag_i -> tag_j
    emit_counts  = defaultdict(Counter)  # tag   -> word
    tag_counts   = Counter()             # tokens per tag
    vocab = set()

    for sent in train_sents:
        prev = None
        for pos, (word, tag) in enumerate(sent):
            word = word.lower()          # case-fold to shrink OOV rate
            vocab.add(word)
            tag_counts[tag] += 1
            emit_counts[tag][word] += 1
            if pos == 0:
                init_counts[tag] += 1
            else:
                trans_counts[prev][tag] += 1
            prev = tag

    tags  = sorted(tag_counts)
    tag2i = {t: i for i, t in enumerate(tags)}
    N, V, n_sents = len(tags), len(vocab), len(train_sents)

    # pi
    log_pi = np.array([np.log(init_counts[t] / n_sents + EPS) for t in tags])

    # A
    log_A = np.empty((N, N))
    for ti, i in tag2i.items():
        denom = sum(trans_counts[ti].values())
        for tj, j in tag2i.items():
            p = (trans_counts[ti][tj] / denom) if denom else 0.0
            log_A[i, j] = np.log(p + EPS)

    # B (Laplace-smoothed emissions, with explicit <OOV> mass)
    log_B = {}
    for tag in tags:
        denom = tag_counts[tag] + alpha * V
        d = {w: np.log((c + alpha) / denom + EPS) for w, c in emit_counts[tag].items()}
        d['<OOV>'] = np.log(alpha / denom + EPS)
        log_B[tag] = d

    return tags, tag2i, log_pi, log_A, log_B, vocab

tags, tag2i, log_pi, log_A, log_B, vocab = train_hmm(train_sents, alpha=1.0)
print("states:", tags)
print(f"|V| = {len(vocab):,}")

states: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
|V| = 45,153


## Step 3 — Log-Space Viterbi Decoder

Multiplying many probabilities in $[0,1]$ underflows to $0.0$ for normal-length
sentences. Working in log-space turns products into sums, so nothing underflows:

$$\log v_t(j)=\max_{i}\big[\log v_{t-1}(i)+\log P(s_j\mid s_i)\big]+\log P(o_t\mid s_j)$$

The $N\times N$ recurrence is vectorised with numpy; backtracking pointers
recover the single best tag sequence.

In [4]:
def log_viterbi(sentence, tags, tag2i, log_pi, log_A, log_B):
    N, T = len(tags), len(sentence)

    def emit_col(word):
        word = word.lower()
        col = np.empty(N)
        for tag, i in tag2i.items():
            tbl = log_B[tag]
            col[i] = tbl[word] if word in tbl else tbl['<OOV>']   # OOV fallback
        return col

    log_delta = np.full((T, N), -np.inf)
    backptr   = np.zeros((T, N), dtype=int)

    log_delta[0] = log_pi + emit_col(sentence[0])               # initialisation
    for t in range(1, T):                                       # recursion
        scores = log_delta[t-1][:, None] + log_A                # scores[i,j]
        backptr[t]   = scores.argmax(axis=0)
        log_delta[t] = scores.max(axis=0) + emit_col(sentence[t])

    path = np.zeros(T, dtype=int)                               # backtrack
    path[T-1] = int(log_delta[T-1].argmax())
    for t in range(T-1, 0, -1):
        path[t-1] = backptr[t, path[t]]
    return [tags[i] for i in path]

# sanity check on a famously ambiguous word
demo = "I watch the dog".split()
print(demo, "->", log_viterbi(demo, tags, tag2i, log_pi, log_A, log_B))

['I', 'watch', 'the', 'dog'] -> ['PRON', 'VERB', 'DET', 'NOUN']


## Step 4 — Evaluation

Word-level accuracy on the held-out 20%:

$$\text{Accuracy}=\frac{\text{correctly tagged words}}{\text{total words}}$$

In [5]:
def evaluate(test_sents, tags, tag2i, log_pi, log_A, log_B):
    total = correct = 0
    oov_total = oov_correct = 0
    confusion = defaultdict(Counter)
    vocab_words = set(w for tbl in log_B.values() for w in tbl) - {'<OOV>'}
    errors = []
    for sent in test_sents:
        words = [w for w, _ in sent]
        gold  = [t for _, t in sent]
        pred  = log_viterbi(words, tags, tag2i, log_pi, log_A, log_B)
        for w, g, p in zip(words, gold, pred):
            total += 1
            is_oov = w.lower() not in vocab_words
            oov_total += is_oov
            if g == p:
                correct += 1
                oov_correct += is_oov
            confusion[g][p] += 1
        if pred != gold:
            errors.append((words, gold, pred))
    return dict(accuracy=correct/total, total=total, correct=correct,
                oov_total=oov_total, oov_acc=oov_correct/oov_total,
                confusion=confusion, errors=errors)

res = evaluate(test_sents, tags, tag2i, log_pi, log_A, log_B)
print(f"WORD-LEVEL ACCURACY (alpha=1.0): {res['accuracy']*100:.2f}%"
      f"  ({res['correct']:,}/{res['total']:,})")
print(f"OOV words: {res['oov_total']:,}   OOV accuracy: {res['oov_acc']*100:.2f}%")

WORD-LEVEL ACCURACY (alpha=1.0): 93.80%  (217,776/232,177)
OOV words: 5,050   OOV accuracy: 41.29%


## Experiment — Effect of the smoothing constant α

The brief uses α = 1.0. With a 45k-word vocabulary that **over-smooths**:
α·|V| dominates the emission denominator. Sweeping α reveals a real trade-off —
smaller α sharpens *known-word* emissions (raising overall accuracy) but starves
the *OOV* mass (lowering OOV accuracy).

In [6]:
print(f"{'alpha':>8} | {'overall':>8} | {'OOV':>7}")
for a in [0.0001, 0.001, 0.01, 0.1, 1.0]:
    p = train_hmm(train_sents, alpha=a)
    r = evaluate(test_sents, *p[:5] if False else (p[0], p[1], p[2], p[3], p[4]))
    print(f"{a:>8} | {r['accuracy']*100:7.2f}% | {r['oov_acc']*100:6.2f}%")

   alpha |  overall |     OOV


  0.0001 |   95.63% |  26.95%


   0.001 |   95.63% |  27.07%


    0.01 |   95.60% |  27.25%


     0.1 |   95.42% |  30.65%


     1.0 |   93.80% |  41.29%


## Error Analysis — top confusions & first-order Markov failures

In [7]:
# rebuild best model
tags, tag2i, log_pi, log_A, log_B, vocab = train_hmm(train_sents, alpha=0.001)
res = evaluate(test_sents, tags, tag2i, log_pi, log_A, log_B)

pairs = Counter()
for g, ctr in res['confusion'].items():
    for p, c in ctr.items():
        if g != p:
            pairs[(g, p)] += c
print("Top confusions (gold -> predicted):")
for (g, p), c in pairs.most_common(8):
    print(f"  {g:>5} -> {p:<5}: {c:,}")

print("\nLong-distance (first-order Markov) failures:")
for s in ["In the physical sciences , these achievements concern electricity , chemistry , and atomic physics .".split(),
          "In wage negotiations , the industry bargains as a unit with a single union .".split()]:
    pred = log_viterbi(s, tags, tag2i, log_pi, log_A, log_B)
    print("  " + " ".join(s))
    for w, p in zip(s, pred):
        if w in ("concern", "bargains"):
            print(f"     '{w}' -> predicted {p} (gold VERB)")

Top confusions (gold -> predicted):
   VERB -> NOUN : 1,039
   NOUN -> ADJ  : 782
   NOUN -> VERB : 625
    ADJ -> ADV  : 548
   NOUN -> X    : 539
   NOUN -> NUM  : 466
    ADV -> ADJ  : 451
    PRT -> ADP  : 416

Long-distance (first-order Markov) failures:
  In the physical sciences , these achievements concern electricity , chemistry , and atomic physics .
     'concern' -> predicted NOUN (gold VERB)
  In wage negotiations , the industry bargains as a unit with a single union .
     'bargains' -> predicted NOUN (gold VERB)
